# 14 — 3D terrain reasoning with Astra and GeoLibre

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/14_astra_terrain_3d.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/14_astra_terrain_3d.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `7762f3bc30d0c1301db9ec2af9668a3c1bbadcadde440f92e13a275a3f79ffec`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 14 — 3D terrain reasoning with Astra and GeoLibre

[Launch in JupyterLite](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=14_astra_terrain_3d.ipynb)

Read one synthetic landscape as contours, a slope field, a profile, a rotatable surface and native GeoLibre extrusions. Use Astra to compare what the representations reveal. **Time:** about 25 minutes. **Prerequisites:** array indexing; lab 00.

**Runtime:** browser Python for the analysis; hosted GeoLibre for maps; optional CPython runner for Astra. The default run makes no model calls. The first run needs network access for packages, GeoLibre and interactive Plotly scripts. Static plots and local analysis do not depend on data APIs.

**Learning cycle:** predict → run → compare views → explain → test a different assumption.

## Start here: the question behind the code

**How do contours, profiles and 3D help us understand terrain?**

**What you will learn:** construct → differentiate → link views → question. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Spatial analysis with optional model interpretation: derivatives measure shape; no terrain-learning model is fitted here.

**Why this is useful:** A browser can make geometry and camera choices tangible, helping people question a terrain explanation before acting on it.

**Where the work happens:** The invented surface, slope and views are computed locally. Plotly and GeoLibre displays use their web rendering resources.

![One surface, three useful views. Use local metre coordinates to define a known teaching surface. Calculate how elevation changes over distance, then express slope in degrees. Trace a fixed cross-section through the same surface shown in contours and 3D. Change vertical exaggeration and see why camera appearance is not a new measurement.](assets/learning/14/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: One surface, three useful views — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/14/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/14/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/14/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=14_astra_terrain_3d.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/14_astra_terrain_3d.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/14/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Construct (0–8 s):** Use local metre coordinates to define a known teaching surface. Code landmark: `x, y = np.meshgrid(axis_m, axis_m)`.

2. **Differentiate (8–16 s):** Calculate how elevation changes over distance, then express slope in degrees. Code landmark: `np.gradient(z, 100, 100)`.

3. **Link views (16–24 s):** Trace a fixed cross-section through the same surface shown in contours and 3D. Code landmark: `z[profile_row]`.

4. **Question (24–32 s):** Change vertical exaggeration and see why camera appearance is not a new measurement. Code landmark: `scene.aspectratio.z`.


## See the connections

![Terrain: 2D, 3D and a profile tell one story: Surface: An analytic terrain uses local metric axes; Derivatives: Slope follows elevation change per metre; 3D maps: GeoLibre extrusions and a rotatable surface; Astra: Compare views without inventing measured terrain](assets/14_mindmap.svg)

### Code guide 1 — Prepare the browser toolbox

**Read the Python:** `import` makes a package available to Python. `sys.platform` identifies the runtime; the indented `if` block runs only in Pyodide. `await` waits for package loading. `LiteMap as Map` gives the browser-compatible map class a shorter name. Installing a package and importing its functions are separate steps.

**Look for:** Imports complete before you run the next cell. The first run can take longer while packages download.

**Pause and try:** Find the alias `Map`. If a later cell says a name is undefined, rerun this setup cell first.


In [ ]:
import sys
if sys.platform == "emscripten":
    from pyodide import loadPackage
    await loadPackage(["numpy", "pandas", "matplotlib", "scipy", "pillow"])
    import micropip
    await micropip.install(["geolibre==3.0.0", "plotly==6.5.0"])
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from geolibre_lite import LiteMap as Map
from cognition import style_plots, COLORS, save_json
from astra_lab import make_request, response_text
style_plots()

## 1. Build a terrain with known units

This surface is an analytic teaching function, **not a DEM of the displayed place**. Local x and y are metres; z is invented elevation in metres. Grid spacing is 100 m. Slope uses the metric spacing, not longitude/latitude degrees.

### Code guide 2 — Distinguish elevation, slope and profile

**Read the Python:** `meshgrid` creates east/north coordinates. Two smooth bumps define an invented elevation surface. `gradient(z,100,100)` uses metre spacing to estimate derivatives; arctangent converts their combined rise/run into a slope angle. The selected array row defines a fixed northing for the profile.

**Look for:** Contour, slope and cross-section views of exactly the same synthetic terrain.

**Pause and try:** Find a high flat area and a lower steep area. Why are elevation and slope different questions?


In [ ]:
from cognition import grid_features, plotly_view, figure_data_url
import plotly.graph_objects as go
axis_m = np.linspace(0,4000,41)
x, y = np.meshgrid(axis_m,axis_m)
z = 100 + 650*np.exp(-((x-1400)**2+(y-2400)**2)/1_100_000) + 350*np.exp(-((x-3100)**2+(y-900)**2)/650_000)
dzdy, dzdx = np.gradient(z,100,100)
slope = np.degrees(np.arctan(np.hypot(dzdx,dzdy)))
profile_row = 24
fig, panels = plt.subplot_mosaic([["contours","slope"],["profile","profile"]],figsize=(10,8),layout="constrained")
axes = [panels[k] for k in ["contours","slope","profile"]]
contours = axes[0].contourf(x/1000,y/1000,z,levels=12,cmap="terrain")
axes[0].axhline(axis_m[profile_row]/1000,color="#dc2626",ls="--",label="Profile")
axes[0].set(xlabel="East (km)",ylabel="North (km)",title="Synthetic elevation contours")
axes[0].legend()
fig.colorbar(contours,ax=axes[0],label="Elevation (m)")
slopes = axes[1].imshow(slope,origin="lower",extent=(0,4,0,4),cmap="magma")
axes[1].set(xlabel="East (km)",ylabel="North (km)",title="Slope is steepest beside the peak")
fig.colorbar(slopes,ax=axes[1],label="Slope (degrees)")
axes[2].plot(axis_m/1000,z[profile_row],color="#0f766e")
axes[2].set(xlabel="East along profile (km)",ylabel="Elevation (m)",title="Cross-section at northing 2.4 km")
terrain_image = figure_data_url(fig)
plt.show()

## 2. Rotate the landscape, then question the camera

Drag to rotate, scroll to zoom, and use the scale menu to compare vertical exaggeration. The surface has the same units on all three axes. A fixed static 3D view below remains readable when the Plotly CDN is unavailable.

### Code guide 3 — Rotate the terrain and inspect exaggeration

**Read the Python:** Plotly's Surface uses the same x, y and z arrays. The aspect ratio compares elevation range to horizontal extent. The menu changes vertical display scale without modifying elevations. A static Matplotlib surface supports reading outside the interactive viewer.

**Look for:** An interactive surface with 1×, 2× and 4× vertical-scale options, plus a static 3D view.

**Pause and try:** Toggle 4× and explain why the computed slope values do not change.


In [ ]:
surface = go.Figure(go.Surface(x=x,y=y,z=z,colorscale="Earth",colorbar={"title":"Elevation (m)"}))
surface.update_layout(title="Synthetic terrain — equal metric axes (1×)",height=530,
    scene={"xaxis_title":"East (m)","yaxis_title":"North (m)","zaxis_title":"Elevation (m)",
           "aspectmode":"manual","aspectratio":{"x":1,"y":1,"z":float(np.ptp(z)/4000)}},
    updatemenus=[{"buttons":[{"label":f"{scale}× vertical","method":"relayout","args":[{
        "scene.aspectratio.z":float(np.ptp(z)/4000*scale),"title.text":f"Synthetic terrain — {scale}× vertical exaggeration"}]} for scale in [1,2,4]]}])
display(plotly_view(surface,height=570))
fig = plt.figure(figsize=(10,5),layout="constrained")
ax = fig.add_subplot(projection="3d")
ax.plot_surface(x/1000,y/1000,z/1000,cmap="terrain",linewidth=0)
ax.set(xlabel="East (km)",ylabel="North (km)",zlabel="Elevation (km)",title="Static 3D view — 1× vertical scale")
ax.set_box_aspect((4,4,np.ptp(z)/1000))
ax.set_zticks([.1,.4,.7])
plt.show()

## 3. A genuine GeoLibre 3D map

These columns represent sampled synthetic elevation, not buildings. Heights are in metres; the map uses the GeoLibre 3.0 extrusion fields and a pitched camera. Coarsening to 10 × 10 columns keeps the browser responsive. Use the contour and profile views to resolve occlusion in 3D.

### Code guide 4 — Put sampled heights into a GeoLibre scene

**Read the Python:** The slices sample every fourth grid point to keep the layer small. `grid_features` makes display cells; the extrusion reads height from `elevation_m`. `set_view` tilts and rotates the camera. Columns represent sampled synthetic terrain.

**Look for:** A tilted map of elevation columns and a saved project recipe.

**Pause and try:** Describe the columns as a sampled representation rather than measured buildings or a continuous surveyed surface.


In [ ]:
terrain_fc = grid_features(z[2:40:4,2:40:4],field="elevation_m",cell_m=400)
terrain_map = Map(center=(-122.278,37.917),zoom=12.5,height="580px")
terrain_map.add_geojson(terrain_fc,name="Synthetic elevation columns (metres)",
    extrusionEnabled=True,extrusionHeightProperty="elevation_m",extrusionHeightScale=1,
    extrusionColor="#0f766e",extrusionOpacity=.9,fillOpacity=.25,strokeWidth=.5)
terrain_map.set_view(pitch=60,bearing=-25)
display(terrain_map)
save_json("exports/synthetic_terrain.geolibre.json",terrain_map.to_project())

## 4. Ask Astra to reconcile the views

The evidence includes the actual maximum slope and the profile location, so Astra need not guess measurements from perspective. Ask what can and cannot be inferred about routing or drainage. Neither roads nor a hydrologic model are supplied.

### Code guide 5 — Prepare an evidence-based terrain question

**Read the Python:** `min`, `max` and profile selection calculate numbers directly from arrays. Converting NumPy values to Python floats makes them JSON-safe. The request supplies these measurements and the 2D figure to support a later explanation.

**Look for:** A saved request with exact synthetic terrain statistics and units.

**Pause and try:** Explain which extra information a real drainage or route decision would need beyond this surface.


In [ ]:
evidence={"evidence_id":"terrain_metrics","source":"analytic synthetic function",
          "grid_spacing_m":100,"min_elevation_m":float(z.min()),"max_elevation_m":float(z.max()),
          "max_slope_degrees":float(slope.max()),"profile_northing_m":2400,
          "profile_max_elevation_m":float(z[profile_row].max()),"crs":"local metric teaching grid"}
request=make_request("Compare contours, slope, and profile. Explain the effect of vertical exaggeration. Which extra data would a defensible route or drainage analysis require? Cite terrain_metrics.",evidence,images=[terrain_image])
save_json("exports/terrain_review_request.json",request)

## Optional: run this request with GPT-6 Astra

All calculations above run locally. The exported request has **not been sent** and any example plan is **hand-authored**, not a recorded model response. To obtain a live response:

1. Download the generated request from JupyterLite's file browser (right-click → Download).
2. In a full CPython terminal in this repository, set `OPENAI_API_KEY` in that terminal's environment. Never put the key in a notebook, output, or the static site.
3. Preview with `python scripts/run_astra.py PATH_TO_REQUEST.json`. Send deliberately with `--send --output astra_response.json`. This uses your API account and can incur charges.
4. Upload `astra_response.json` to the notebook folder, set the import path in the next cell, and rerun it. Inspect its evidence claims before using them.

For notebook 15, also download `accessibility_evidence.json` and pass `--evidence PATH_TO_EVIDENCE.json`. The runner only dispatches the one allowlisted local tool, with a bounded number of rounds.

The static JupyterLite site does not host this runner or proxy requests. Model access depends on your API project. Image reading is an aid to interpretation; geometry, distances and metrics come from code and explicit metadata.

### Code guide 6 — Inspect an optional model response

**Read the Python:** `None` means no response file was supplied. Only a real path enters the `if` branch; `json.loads` converts saved JSON text into Python objects. The `else` branch keeps the default lesson usable without an API call. Follow the preceding export/import instructions for live use.

**Look for:** The default run prints an offline message. An imported answer is evidence to inspect, not a new measurement.

**Pause and try:** Write your own explanation first, then check any imported response against the computed values and units.


In [ ]:
RESPONSE_PATH=None
if RESPONSE_PATH:
    display(json.loads(response_text(json.loads(Path(RESPONSE_PATH).read_text()))))
else:
    print("Offline terrain evidence ready; no model response imported.")

## 5. Transfer to measured terrain

Open lab 00's real DEM in GeoLibre and inspect its metadata. GeoLibre's **Controls → Terrain** can show public terrain context; that control does not automatically turn every loaded COG into the scene's elevation source. For quantitative analysis, obtain the actual DEM values, horizontal CRS, vertical datum and nodata mask in a full Python runtime. Compute slope in metric units and record any resampling.

**Predict → test:** double the synthetic elevation above its base. What happens to slope? Rotate the 3D view 180 degrees. Does the underlying terrain change, or only the visibility of its slopes?

## Data & software citations

- Teaching fixtures: deterministic synthetic arrays and networks authored in this repository. Values, scenes, facilities and outcomes are invented; coordinates only anchor the display. No actual people, environmental measurements, or service areas are represented.
- [GeoLibre](https://geolibre.app/) and [GeoLibre map controls](https://geolibre.app/user-guide/map-controls/); `geolibre==3.0.0` project builders through `geolibre_lite.LiteMap`.
- [GeoAI](https://opengeoai.org/): full Python/GPU continuation for imagery models. These browser labs implement workflow patterns; they do not install or run `geoai-py`.
- [GPT-6 Astra model](https://developers.openai.com/api/docs/models/gpt-6-astra), [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs), [image inputs](https://developers.openai.com/api/docs/guides/images-vision), and [function calling](https://developers.openai.com/api/docs/guides/function-calling). API examples checked 2026-10-02.
- [Pyodide packages](https://pyodide.org/en/stable/usage/packages-in-pyodide.html), [Matplotlib](https://matplotlib.org/stable/), [Plotly Python](https://plotly.com/python/).
- GeoLibre basemap tiles retain their provider attribution, including [OpenStreetMap contributors](https://www.openstreetmap.org/copyright) where applicable.

See `DATA_SOURCES.md` for the source register and runtime boundaries.

## Make sense of the result

![The 4x mountain looks much more dangerous! Only the display scale changed; the slopes did not. I will read units and compare the profile.](assets/learning/14/cartoon.svg)

**Cartoon transcript:** The 4x mountain looks much more dangerous! → Only the display scale changed; the slopes did not. → I will read units and compare the profile.

**Explain it to a colleague:** Can the highest cell have a gentler slope than a cell on the hillside?

<details><summary>Check your reasoning</summary>

Yes. Elevation describes height; slope describes local change in height over distance.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
